<a href="https://colab.research.google.com/github/tylerdurdenhere/db_and_analytics/blob/main/sql_in_r.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Install & Load Libraries
install.packages("sqldf")
install.packages("ggplot2")
install.packages("dplyr")
install.packages("readr")
install.packages("knitr")
install.packages("scales")

library(sqldf)
library(ggplot2)
library(dplyr)
library(readr)
library(knitr)
library(scales)

cat("All libraries loaded successfully.\n")

Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

also installing the dependencies ‘gsubfn’, ‘proto’, ‘RSQLite’, ‘chron’


Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

Loading required package: gsubfn

Loading required package: proto

Warning message:
“no DISPLAY variable so Tk is not available”
Loading required package: RSQLite


Attaching package: ‘dplyr’


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union



Attaching package: ‘scales’


The following object is masked fr

All libraries loaded successfully.


In [11]:
system("pip install google-colab", intern=TRUE)

[1] "Requirement already satisfied: google-colab in /usr/local/lib/python3.12/dist-packages (1.0.0)"                                                                                                                                                     
 [2] "Requirement already satisfied: google-auth==2.47.0 in /usr/local/lib/python3.12/dist-packages (from google-colab) (2.47.0)"                                                                                                                         
 [3] "Requirement already satisfied: ipykernel==6.17.1 in /usr/local/lib/python3.12/dist-packages (from google-colab) (6.17.1)"                                                                                                                           
 [4] "Requirement already satisfied: ipyparallel==8.8.0 in /usr/local/lib/python3.12/dist-packages (from google-colab) (8.8.0)"                                                                                                                           
 [5] "Requirement already satisfied: ipython==7.34.0 in /usr/local/lib/python3.12/dist-packages (from google-colab) (7.34.0)"                                                                                                                             
 [6] "Requirement already satisfied: pandas==2.2.2 in /usr/local/lib/python3.12/dist-packages (from google-colab) (2.2.2)"                                                                                                                                
 [7] "Requirement already satisfied: jupyter-server==2.14.0 in /usr/local/lib/python3.12/dist-packages (from google-colab) (2.14.0)"                                                                                                                      
 [8] "Requirement already satisfied: portpicker==1.5.2 in /usr/local/lib/python3.12/dist-packages (from google-colab) (1.5.2)"                                                                                                                            
 [9] "Requirement already satisfied: requests==2.32.4 in /usr/local/lib/python3.12/dist-packages (from google-colab) (2.32.4)"                                                                                                                            
[10] "Requirement already satisfied: tornado==6.5.1 in /usr/local/lib/python3.12/dist-packages (from google-colab) (6.5.1)"                                                                                                                               
[11] "Requirement already satisfied: pyasn1-modules>=0.2.1 in /usr/local/lib/python3.12/dist-packages (from google-auth==2.47.0->google-colab) (0.4.2)"                                                                                                   
[12] "Requirement already satisfied: rsa<5,>=3.1.4 in /usr/local/lib/python3.12/dist-packages (from google-auth==2.47.0->google-colab) (4.9.1)"                                                                                                           
[13] "Requirement already satisfied: debugpy>=1.0 in /usr/local/lib/python3.12/dist-packages (from ipykernel==6.17.1->google-colab) (1.8.15)"                                                                                                             
[14] "Requirement already satisfied: jupyter-client>=6.1.12 in /usr/local/lib/python3.12/dist-packages (from ipykernel==6.17.1->google-colab) (7.4.9)"                                                                                                    
[15] "Requirement already satisfied: matplotlib-inline>=0.1 in /usr/local/lib/python3.12/dist-packages (from ipykernel==6.17.1->google-colab) (0.2.1)"                                                                                                    
[16] "Requirement already satisfied: nest-asyncio in /usr/local/lib/python3.12/dist-packages (from ipykernel==6.17.1->google-colab) (1.6.0)"                                                                                                

In [23]:
# Load Cleaned Datasets

orders     <- read_csv("orders_clean.csv")
deliveries <- read_csv("deliveries_clean.csv")
customers  <- read_csv("customers_clean.csv")
drivers    <- read_csv("drivers_clean.csv")
vehicles   <- read_csv("vehicles_clean.csv")
hubs       <- read_csv("hubs_clean.csv")
incidents  <- read_csv("incidents_clean.csv")
complaints <- read_csv("complaints_clean.csv")
app_events <- read_csv("app_events_clean.csv")

cat("Datasets loaded:\n")
cat(sprintf("  orders     : %d rows\n", nrow(orders)))
cat(sprintf("  deliveries : %d rows\n", nrow(deliveries)))
cat(sprintf("  customers  : %d rows\n", nrow(customers)))
cat(sprintf("  drivers    : %d rows\n", nrow(drivers)))
cat(sprintf("  vehicles   : %d rows\n", nrow(vehicles)))
cat(sprintf("  hubs       : %d rows\n", nrow(hubs)))
cat(sprintf("  incidents  : %d rows\n", nrow(incidents)))
cat(sprintf("  complaints : %d rows\n", nrow(complaints)))
cat(sprintf("  app_events : %d rows\n", nrow(app_events)))

Rows: 1250 Columns: 14
── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
chr  (9): order_id, customer_id, service_type, pickup_zone, dropoff_zone, pr...
dbl  (4): promised_window_hours, order_value, special_handling_flag, order_hour
dttm (1): order_created_at

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.
Rows: 950 Columns: 16
── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
chr  (6): delivery_id, order_id, driver_id, vehicle_id, hub_id, delivery_status
dbl  (8): route_distance_km, manual_route_override_count, proof_of_completio...
dttm (2): dispatch_time, delivery_completed_at

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.
Rows: 650 Columns: 10
── Column specification ─────────────

Datasets loaded:
  orders     : 1250 rows
  deliveries : 950 rows
  customers  : 650 rows
  drivers    : 170 rows
  vehicles   : 120 rows
  hubs       : 8 rows
  incidents  : 280 rows
  complaints : 320 rows
  app_events : 640 rows


In [24]:
# SQL Query 1 — Delivery Failure Rate by Zone
# Business question: Which pickup zones have the highest failure and delay rates?

q1 <- sqldf("
  SELECT
    o.pickup_zone,
    COUNT(d.delivery_id)                                         AS total_deliveries,
    SUM(CASE WHEN d.delivery_status IN ('Delayed','Failed')
             THEN 1 ELSE 0 END)                                  AS failed_or_delayed,
    ROUND(
      SUM(CASE WHEN d.delivery_status IN ('Delayed','Failed')
               THEN 1.0 ELSE 0 END) / COUNT(d.delivery_id) * 100
    , 2)                                                         AS failure_rate_pct,
    ROUND(AVG(d.customer_rating_post_delivery), 2)               AS avg_customer_rating,
    ROUND(AVG(d.manual_route_override_count), 2)                 AS avg_overrides
  FROM deliveries d
  JOIN orders o ON d.order_id = o.order_id
  GROUP BY o.pickup_zone
  ORDER BY failure_rate_pct DESC
")

print(q1)
cat("\nInterpretation: Zones with failure rates above 30% require urgent operational review.\n")
cat("High override averages in failing zones suggest poor route planning rather than driver error.\n")

  pickup_zone total_deliveries failed_or_delayed failure_rate_pct
1     Central              174                84            48.28
2     Airport              113                43            38.05
3   Riverside              119                43            36.13
4        East              156                50            32.05
5       North              135                43            31.85
6        West              114                35            30.70
7       South              139                36            25.90
  avg_customer_rating avg_overrides
1                3.56          1.29
2                3.99          1.81
3                3.87          0.73
4                3.91          0.79
5                3.90          0.70
6                3.90          0.81
7                4.05          0.69

Interpretation: Zones with failure rates above 30% require urgent operational review.
High override averages in failing zones suggest poor route planning rather than driver error.


In [25]:
# SQL Query 2 — Hub Performance Ranking
# Business question: Which hubs are underperforming across multiple metrics?

q2 <- sqldf("
  SELECT
    h.hub_name,
    h.zone,
    h.hub_type,
    COUNT(d.delivery_id)                                         AS total_dispatches,
    SUM(CASE WHEN d.delivery_status = 'Failed'
             THEN 1 ELSE 0 END)                                  AS total_failures,
    SUM(CASE WHEN d.delivery_status = 'Delayed'
             THEN 1 ELSE 0 END)                                  AS total_delays,
    ROUND(
      SUM(CASE WHEN d.delivery_status IN ('Delayed','Failed')
               THEN 1.0 ELSE 0 END) / COUNT(d.delivery_id) * 100
    , 2)                                                         AS failure_rate_pct,
    ROUND(AVG(d.customer_rating_post_delivery), 2)               AS avg_rating,
    ROUND(AVG(d.manual_route_override_count), 2)                 AS avg_overrides
  FROM deliveries d
  JOIN hubs h ON d.hub_id = h.hub_id
  GROUP BY h.hub_name, h.zone, h.hub_type
  ORDER BY failure_rate_pct DESC
")

print(q2)
cat("\nInterpretation: Hubs with high failure rates AND low ratings indicate systemic operational issues.\n")
cat("Cross-referencing hub_type reveals whether logistics or mobility hubs are more problematic.\n")

        hub_name      zone  hub_type total_dispatches total_failures
1   Central Core   Central   Control              115             23
2    Airport Hub   Airport  Dispatch              104             15
3  Midtown Relay   Central  Charging              128             26
4      West Gate      West  Dispatch              127             16
5     South Link     South  Dispatch              106             10
6  Riverside Hub Riverside Warehouse              115             14
7 North Exchange     North  Dispatch              136             17
8      East Dock      East Warehouse              119             11
  total_delays failure_rate_pct avg_rating avg_overrides
1           25            41.74       3.68          0.95
2           27            40.38       3.88          0.91
3           22            37.50       3.89          1.11
4           28            34.65       3.92          0.87
5           26            33.96       3.95          0.92
6           25            33.91      

In [26]:
# SQL Query 3 — Driver Performance Analysis
# Business question: Which drivers have the highest override and failure rates?

q3 <- sqldf("
  SELECT
    d.driver_id,
    dr.employment_type,
    dr.base_zone,
    dr.driver_rating,
    COUNT(d.delivery_id)                                         AS total_deliveries,
    SUM(CASE WHEN d.delivery_status IN ('Delayed','Failed')
             THEN 1 ELSE 0 END)                                  AS failures,
    ROUND(
      SUM(CASE WHEN d.delivery_status IN ('Delayed','Failed')
               THEN 1.0 ELSE 0 END) / COUNT(d.delivery_id) * 100
    , 2)                                                         AS failure_rate_pct,
    ROUND(AVG(d.manual_route_override_count), 2)                 AS avg_overrides,
    ROUND(AVG(d.customer_rating_post_delivery), 2)               AS avg_customer_rating
  FROM deliveries d
  JOIN drivers dr ON d.driver_id = dr.driver_id
  GROUP BY d.driver_id, dr.employment_type, dr.base_zone, dr.driver_rating
  HAVING total_deliveries >= 5
  ORDER BY failure_rate_pct DESC
  LIMIT 15
")

print(q3)
cat("\nInterpretation: Drivers with high failure rates and high overrides may need retraining or reassignment.\n")
cat("Employment type comparison reveals if contract drivers underperform relative to full-time staff.\n")

   driver_id employment_type base_zone driver_rating total_deliveries failures
1       D100        FullTime   Central          4.31                8        6
2       D053        FullTime      West          3.80                7        5
3       D023        FullTime      East          4.16                6        4
4       D141        PartTime     North          3.44                9        6
5       D165        PartTime     North          3.89                6        4
6       D005        FullTime     North          4.14                5        3
7       D092        FullTime      East          4.24                5        3
8       D094        PartTime   Central          4.48                5        3
9       D095        FullTime      West          3.15                5        3
10      D156        FullTime     South          3.62                5        3
11      D162        Contract Riverside          4.62                5        3
12      D168        FullTime     North          3.84

In [27]:
# SQL Query 4 — Customer Complaint Patterns
# Business question: Which customer segments and zones generate the most severe complaints?

q4 <- sqldf("
  SELECT
    c.home_zone,
    c.customer_type,
    cp.complaint_type,
    cp.severity,
    COUNT(cp.complaint_id)                                       AS complaint_count,
    ROUND(AVG(cp.compensation_amount), 2)                        AS avg_compensation,
    SUM(CASE WHEN cp.status = 'Resolved'
             THEN 1 ELSE 0 END)                                  AS resolved_count,
    ROUND(
      SUM(CASE WHEN cp.status = 'Resolved'
               THEN 1.0 ELSE 0 END) / COUNT(cp.complaint_id) * 100
    , 2)                                                         AS resolution_rate_pct
  FROM complaints cp
  JOIN orders o   ON cp.order_id   = o.order_id
  JOIN customers c ON o.customer_id = c.customer_id
  GROUP BY c.home_zone, c.customer_type, cp.complaint_type, cp.severity
  ORDER BY complaint_count DESC
  LIMIT 20
")

print(q4)
cat("\nInterpretation: High severity complaints with low resolution rates identify service gaps.\n")
cat("Business customers with repeated complaints signal contract risk for NorthStar.\n")

   home_zone customer_type  complaint_type severity complaint_count
1    Airport      Consumer           Delay   Medium               8
2  Riverside      Consumer           Delay   Medium               8
3      South      Consumer    MissedPickup   Medium               8
4      North      Consumer        AppIssue   Medium               7
5      North      Consumer           Delay   Medium               7
6      North      Consumer    MissedPickup   Medium               7
7    Central      Consumer DriverBehaviour   Medium               6
8       East      Consumer           Delay   Medium               6
9      South      Consumer           Delay      Low               6
10     South      Consumer           Delay   Medium               6
11   Central      Consumer           Delay   Medium               5
12   Central      Consumer    MissedPickup   Medium               5
13     North      Consumer    MissedPickup      Low               5
14      West      Consumer           Delay   Med

In [28]:
# SQL Query 5 — Route Profitability Analysis
# Business question: Which route combinations (pickup → dropoff) are most problematic?

q5 <- sqldf("
  SELECT
    o.pickup_zone,
    o.dropoff_zone,
    o.service_type,
    COUNT(o.order_id)                                            AS total_orders,
    ROUND(AVG(o.order_value), 2)                                 AS avg_order_value,
    SUM(CASE WHEN d.delivery_status IN ('Delayed','Failed')
             THEN 1 ELSE 0 END)                                  AS failures,
    ROUND(
      SUM(CASE WHEN d.delivery_status IN ('Delayed','Failed')
               THEN 1.0 ELSE 0 END) / COUNT(o.order_id) * 100
    , 2)                                                         AS failure_rate_pct,
    ROUND(AVG(d.customer_rating_post_delivery), 2)               AS avg_rating
  FROM orders o
  JOIN deliveries d ON o.order_id = d.order_id
  GROUP BY o.pickup_zone, o.dropoff_zone, o.service_type
  HAVING total_orders >= 10
  ORDER BY failure_rate_pct DESC
  LIMIT 20
")

print(q5)
cat("\nInterpretation: High-failure routes with low average order value are candidates for discontinuation.\n")
cat("Cross-zone routes with consistent failures suggest infrastructure rather than driver issues.\n")

  pickup_zone dropoff_zone service_type total_orders avg_order_value failures
1     Central      Airport       Retail           10          113.23        8
2        East        South    Passenger           11           86.76        5
3     Central      Central    Passenger           10           88.61        4
4   Riverside      Central       Retail           10          117.48        3
5        East      Central    Passenger           11          113.98        3
6        East    Riverside       Parcel           10           83.62        2
7     Central         West       Retail           10           60.70        1
  failure_rate_pct avg_rating
1            80.00       2.92
2            45.45       4.03
3            40.00       4.01
4            30.00       3.94
5            27.27       3.78
6            20.00       4.25
7            10.00       4.07

Interpretation: High-failure routes with low average order value are candidates for discontinuation.
Cross-zone routes with consistent 

In [29]:
# SQL Query 6 — Vehicle Maintenance and Incident Correlation
# Business question: Are poorly maintained vehicles generating more incidents?

q6 <- sqldf("
  SELECT
    v.vehicle_id,
    v.vehicle_type,
    v.maintenance_status,
    v.assigned_zone,
    ROUND(v.battery_health_pct, 1)                               AS battery_health_pct,
    COUNT(d.delivery_id)                                         AS total_deliveries,
    COUNT(i.incident_id)                                         AS total_incidents,
    ROUND(
      COUNT(i.incident_id) * 1.0 / COUNT(d.delivery_id) * 100
    , 2)                                                         AS incident_rate_pct,
    SUM(CASE WHEN d.delivery_status IN ('Delayed','Failed')
             THEN 1 ELSE 0 END)                                  AS delivery_failures
  FROM vehicles v
  JOIN deliveries d  ON v.vehicle_id  = d.vehicle_id
  LEFT JOIN incidents i ON d.delivery_id = i.delivery_id
  GROUP BY v.vehicle_id, v.vehicle_type, v.maintenance_status,
           v.assigned_zone, v.battery_health_pct
  ORDER BY incident_rate_pct DESC
  LIMIT 15
")

print(q6)
cat("\nInterpretation: Vehicles with 'Overdue' maintenance status and low battery health show higher incident rates.\n")
cat("Proactive maintenance scheduling for high-incident vehicles would reduce operational disruption.\n")

   vehicle_id vehicle_type maintenance_status assigned_zone battery_health_pct
1        V003     CargoVan             Active         North               91.7
2        V076       Diesel           InRepair       Central               65.8
3        V019       Diesel          Scheduled         South               70.4
4        V089     CargoVan           InRepair       Central               93.6
5        V097           EV             Active       Central               92.1
6        V108       Diesel           InRepair       Airport               54.6
7        V112           EV           InRepair         South               94.1
8        V035     CargoVan             Active          East               83.6
9        V046           EV             Active         North               95.8
10       V009     CargoVan             Active         South               68.8
11       V025       Diesel             Active       Airport               42.0
12       V039           EV           InRepair       

In [30]:
# SQL Query 7 — High Value Customers with Service Failures
# Business question: Are high-value customers experiencing disproportionate failures?
# This query demonstrates optimised filtering using indexed key columns

q7 <- sqldf("
  SELECT
    c.customer_id,
    c.customer_type,
    c.home_zone,
    ROUND(c.loyalty_score, 1)                                    AS loyalty_score,
    COUNT(o.order_id)                                            AS total_orders,
    ROUND(SUM(o.order_value), 2)                                 AS total_spend,
    SUM(CASE WHEN d.delivery_status IN ('Delayed','Failed')
             THEN 1 ELSE 0 END)                                  AS failed_deliveries,
    COUNT(cp.complaint_id)                                       AS total_complaints,
    ROUND(
      SUM(CASE WHEN d.delivery_status IN ('Delayed','Failed')
               THEN 1.0 ELSE 0 END) / COUNT(o.order_id) * 100
    , 2)                                                         AS failure_rate_pct
  FROM customers c
  JOIN orders o       ON c.customer_id  = o.customer_id
  JOIN deliveries d   ON o.order_id     = d.order_id
  LEFT JOIN complaints cp ON o.order_id = cp.order_id
  GROUP BY c.customer_id, c.customer_type, c.home_zone, c.loyalty_score
  HAVING total_orders >= 3
     AND total_spend   > 200
  ORDER BY failure_rate_pct DESC, total_spend DESC
  LIMIT 20
")

print(q7)
cat("\nInterpretation: High-spend customers experiencing frequent failures are at churn risk.\n")
cat("These customers should be prioritised for service recovery and proactive communication.\n")

   customer_id customer_type home_zone loyalty_score total_orders total_spend
1        C0109      Consumer     North          52.7            3      313.64
2        C0359      Consumer   Airport          75.7            3      239.91
3        C0056      Consumer     North          71.1            3      203.89
4        C0530           SME      West          55.9            4      477.16
5        C0529           SME Riverside          43.2            4      349.36
6        C0197      Consumer Riverside          58.4            3      444.41
7        C0104      Consumer      East          66.2            3      389.68
8        C0182    Enterprise      East          47.2            3      380.41
9        C0626      Consumer     South          61.6            3      328.06
10       C0575      Consumer Riverside          77.2            3      319.01
11       C0370    Enterprise      East          64.6            3      286.53
12       C0167      Consumer   Central          52.5            

In [31]:
# Visualise SQL Query Results

# Chart 1: Failure rate by zone
png("r_viz1_failure_by_zone.png", width=900, height=500, res=120)
ggplot(q1, aes(x=reorder(pickup_zone, -failure_rate_pct),
               y=failure_rate_pct, fill=failure_rate_pct)) +
  geom_bar(stat="identity", width=0.6) +
  geom_text(aes(label=paste0(failure_rate_pct, "%")),
            vjust=-0.4, size=3.5, fontface="bold") +
  scale_fill_gradient(low="#f1c40f", high="#e74c3c") +
  labs(title="Delivery Failure & Delay Rate by Pickup Zone",
       subtitle="SQL Query 1 — Zones ranked by failure rate",
       x="Pickup Zone", y="Failure Rate (%)") +
  theme_minimal(base_size=12) +
  theme(legend.position="none",
        plot.title=element_text(face="bold"),
        axis.text.x=element_text(angle=30, hjust=1))
dev.off()

# Chart 2: Hub failure rate
png("r_viz2_hub_failure.png", width=900, height=500, res=120)
ggplot(q2, aes(x=reorder(hub_name, -failure_rate_pct),
               y=failure_rate_pct, fill=hub_type)) +
  geom_bar(stat="identity", width=0.6) +
  geom_text(aes(label=paste0(failure_rate_pct, "%")),
            vjust=-0.4, size=3.5, fontface="bold") +
  scale_fill_brewer(palette="Set2") +
  labs(title="Delivery Failure Rate by Hub",
       subtitle="SQL Query 2 — Hubs ranked by failure rate, coloured by hub type",
       x="Hub", y="Failure Rate (%)", fill="Hub Type") +
  theme_minimal(base_size=12) +
  theme(plot.title=element_text(face="bold"),
        axis.text.x=element_text(angle=30, hjust=1))
dev.off()

# Chart 3: Top 10 driver failure rates
top10_drivers <- head(q3, 10)
png("r_viz3_driver_failure.png", width=900, height=500, res=120)
ggplot(top10_drivers, aes(x=reorder(driver_id, -failure_rate_pct),
                           y=failure_rate_pct, fill=employment_type)) +
  geom_bar(stat="identity", width=0.6) +
  geom_text(aes(label=paste0(failure_rate_pct, "%")),
            vjust=-0.4, size=3.2, fontface="bold") +
  scale_fill_brewer(palette="Set1") +
  labs(title="Top 10 Drivers by Delivery Failure Rate",
       subtitle="SQL Query 3 — Coloured by employment type",
       x="Driver ID", y="Failure Rate (%)", fill="Employment Type") +
  theme_minimal(base_size=12) +
  theme(plot.title=element_text(face="bold"),
        axis.text.x=element_text(angle=30, hjust=1))
dev.off()

cat("3 R visualisations saved.\n")
cat("  r_viz1_failure_by_zone.png\n")
cat("  r_viz2_hub_failure.png\n")
cat("  r_viz3_driver_failure.png\n")

agg_record_13191ec332bd 
                      2

agg_record_13191ec332bd 
                      2

agg_record_13191ec332bd 
                      2

3 R visualisations saved.
  r_viz1_failure_by_zone.png
  r_viz2_hub_failure.png
  r_viz3_driver_failure.png
